In [1]:
import os
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BASE_DIR = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge"
CSV_PATH = os.path.join(BASE_DIR, "df_subset_3000.csv")
IMG_DIR  = os.path.join(BASE_DIR, "imagenes-crop-8-3000")

Device: cuda


PRUEBA DE MLFLOW

In [2]:
import mlflow

mlflow.set_experiment("torax-cnn-3000img")

C:\Users\trodr\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='file:///c:/Users/trodr/Documents/proyecto-torax-v2.0/03-notebook/mlruns/4', creation_time=1781979214733, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1781979214733, lifecycle_stage='active', name='torax-cnn-3000img', tags={}, trace_location=None, workspace='default'>

In [3]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())
print("\nPrimeras filas:")
df.head()

Shape: (3000, 20)

Columnas:
['dicom_id', 'subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices', 'full_path', 'ViewPosition', 'calidad-imagen']

Primeras filas:


,dicom_id,subject_id,study_id,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,full_path,ViewPosition,calidad-imagen
0,51adabdf-4234d20c-a63f7056-421c65ab-6dabfa41,16700191,51325054,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,files/p16/p16700191/s51325054/51adabdf-4234d20...,PA,1.0
1,89177690-364bac2f-5d496304-387f233c-db26c1fd,18452091,58217747,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,NaN,NaN,files/p18/p18452091/s58217747/89177690-364bac2...,PA,NaN
2,3b14f064-390ac420-34a6ad8c-351114dd-66126ab3,13742148,58404524,NaN,1.0,1.0,0.0,NaN,NaN,NaN,1.0,NaN,1.0,NaN,-1.0,NaN,1.0,files/p13/p13742148/s58404524/3b14f064-390ac42...,PA,1.0
3,90e66083-301ca562-eed7b130-282a03e8-4903ecc2,17741087,57570419,NaN,1.0,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,1.0,files/p17/p17741087/s57570419/90e66083-301ca56...,AP,NaN
4,5fd80287-eef90ff5-4b97cd1f-263d18f2-ffe8f28c,14121491,50057752,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,files/p14/p14121491/s50057752/5fd80287-eef90ff...,PA,1.0


In [4]:
LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax"
]

# Verificar que las 14 columnas existen
faltantes = [c for c in LABEL_COLS if c not in df.columns]
print("Etiquetas faltantes:", faltantes if faltantes else "ninguna ✓")

# Revisar distribución de positivos por etiqueta (cuántos 1 hay en cada una)
print("\nPositivos por etiqueta:")
print(df[LABEL_COLS].sum().sort_values(ascending=False))

# Revisar valores únicos (deberían ser 0.0 y 1.0 tras U-zeros)
print("\nValores únicos en etiquetas:", np.unique(df[LABEL_COLS].values))

Etiquetas faltantes: ninguna ✓

Positivos por etiqueta:
No Finding                    993.0
Pleural Effusion              619.0
Lung Opacity                  606.0
Cardiomegaly                  511.0
Atelectasis                   459.0
Edema                         201.0
Pneumothorax                  116.0
Consolidation                  77.0
Lung Lesion                    71.0
Fracture                       40.0
Pleural Other                   4.0
Enlarged Cardiomediastinum    -19.0
Pneumonia                     -24.0
dtype: float64

Valores únicos en etiquetas: [-1.  0.  1. nan]


In [5]:
# Filas sin ningún -1 en ningún label
df_clean = df[~(df[LABEL_COLS] == -1).any(axis=1)]
print(f"Filas originales:  {len(df)}")
print(f"Filas sin -1:      {len(df_clean)}")
print(f"Filas eliminadas:  {len(df) - len(df_clean)}")

Filas originales:  3000
Filas sin -1:      2287
Filas eliminadas:  713


In [6]:
for col in LABEL_COLS:
    n_neg1 = (df[col] == -1).sum()
    print(f"  {col:<30} -1: {n_neg1:>4}")

  Atelectasis                    -1:  143
  Cardiomegaly                   -1:   72
  Consolidation                  -1:   64
  Edema                          -1:  177
  Enlarged Cardiomediastinum     -1:  119
  Fracture                       -1:    7
  Lung Lesion                    -1:   12
  Lung Opacity                   -1:   58
  No Finding                     -1:    0
  Pleural Effusion               -1:   85
  Pleural Other                  -1:   13
  Pneumonia                      -1:  244
  Pneumothorax                   -1:   22


In [7]:
#def aplicar_u_zeros(df, label_cols):
 #   df = df.copy()
  #  df[label_cols] = df[label_cols].fillna(0.0)        # NaN -> 0
   # df[label_cols] = df[label_cols].replace(-1.0, 0.0) # incierto -> 0
    #return df

#df = aplicar_u_zeros(df, LABEL_COLS)

#print("Valores únicos tras U-zeros:", np.unique(df[LABEL_COLS].values))

In [8]:
# Construir full_path apuntando a 3000imagenes
df["full_path"] = df["full_path"].apply(
    lambda p: os.path.join(IMG_DIR, p.replace("/", os.sep))
)

# Verificación
existen = df["full_path"].apply(os.path.exists)
print(f"Paths que existen: {existen.sum()} / {len(df)}")
print(f"\nEjemplo: {df['full_path'].iloc[0]}")

if existen.sum() < len(df):
    print(f"\n⚠ Faltan {len(df) - existen.sum()} imágenes. Las dropeamos:")
    df = df[existen].reset_index(drop=True)
    print(f"Filas restantes: {len(df)}")

Paths que existen: 2990 / 3000

Ejemplo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\imagenes-crop-8-3000\files\p16\p16700191\s51325054\51adabdf-4234d20c-a63f7056-421c65ab-6dabfa41.jpg

⚠ Faltan 10 imágenes. Las dropeamos:
Filas restantes: 2990


aqui declaramos el torax-datset , para este de 3000 guardamos las imagenes en la ram

In [9]:
from PIL import Image
from tqdm import tqdm

class ToraxDataset(Dataset):
    def __init__(self, df, label_cols, transform=None):
        self.labels    = torch.tensor(df[label_cols].values, dtype=torch.float32)
        self.transform = transform

        print("Pre-cargando imágenes en RAM...")
        self.images = []
        for path in tqdm(df["full_path"].values):
            img = Image.open(path).convert("RGB").resize((224, 224))
            self.images.append(img.copy())  # .copy() cierra el archivo
        print(f"  {len(self.images)} imágenes en memoria")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [10]:
#class ToraxDataset(Dataset):
 #   def __init__(self, df, label_cols, transform=None):
  #      self.df = df.reset_index(drop=True)
   #     self.label_cols = label_cols
    #    self.transform = transform

    #def __len__(self):
     #   return len(self.df)

    #def __getitem__(self, idx):
     #   row = self.df.iloc[idx]

        # 1. Cargar la imagen desde disco
      #  img = Image.open(row["full_path"]).convert("RGB")

        # 2. Aplicar transformaciones (resize, normalizar, a tensor)
       # if self.transform:
        #    img = self.transform(img)

        # 3. Armar el vector de etiquetas como tensor float
        #labels = torch.tensor(row[self.label_cols].values.astype("float32"))

        #return img, labels

In [11]:
def split_patient_level(df, group_col="subject_id", test_size=0.2, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, val_idx = next(gss.split(df, groups=df[group_col]))
    return df.iloc[train_idx], df.iloc[val_idx]

train_df, val_df = split_patient_level(df)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")
solapan = set(train_df["subject_id"]) & set(val_df["subject_id"])
print("Pacientes solapados (debe ser 0):", len(solapan))

IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ToraxDataset(train_df, LABEL_COLS, train_tf)
val_ds   = ToraxDataset(val_df,   LABEL_COLS, val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=0, pin_memory=True)

imgs, lbls = next(iter(train_loader))
print("\nBatch imágenes:", imgs.shape)   # esperado [32, 3, 224, 224]
print("Batch etiquetas:", lbls.shape)    # esperado [32, 13]

Train: 2397 | Val: 593
Pacientes solapados (debe ser 0): 0
Pre-cargando imágenes en RAM...


100%|██████████| 2397/2397 [01:35<00:00, 25.16it/s]


  2397 imágenes en memoria
Pre-cargando imágenes en RAM...


100%|██████████| 593/593 [00:23<00:00, 24.89it/s]


  593 imágenes en memoria

Batch imágenes: torch.Size([32, 3, 224, 224])
Batch etiquetas: torch.Size([32, 13])


In [12]:
class ToraxCNN(nn.Module):
    def __init__(self, n_classes=13):
        super().__init__()

        # --- Extractor de características ---
        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                # 224 -> 112
            # Bloque 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),                # 112 -> 56
            # Bloque 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),                # 56 -> 28
            # Bloque 4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),                # 28 -> 14
        )

        # --- Clasificador ---
        self.pool = nn.AdaptiveAvgPool2d(1)   # [128, 14, 14] -> [128, 1, 1]
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # [128, 1, 1] -> [128]
            nn.Dropout(0.4),
            nn.Linear(128, n_classes),        # [128] -> [13]
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = ToraxCNN(n_classes=len(LABEL_COLS)).to(DEVICE)

# Verificar shapes con un tensor de prueba
x_prueba = torch.randn(2, 3, 224, 224).to(DEVICE)
salida = model(x_prueba)
print("Entrada:", x_prueba.shape)   # [2, 3, 224, 224]
print("Salida:", salida.shape)      # [2, 13]

Entrada: torch.Size([2, 3, 224, 224])
Salida: torch.Size([2, 13])


In [13]:
train_labels = torch.tensor(
    train_df[LABEL_COLS].fillna(0).values, dtype=torch.float32
)
n_pos = (train_labels == 1).sum(dim=0).clamp(min=1)
n_neg = (train_labels == 0).sum(dim=0).clamp(min=1)
pos_weight = (n_neg / n_pos).to(DEVICE) * 0.75

print("pos_weight por label:")
for name, w in zip(LABEL_COLS, pos_weight.cpu()):
    print(f"  {name:<30} {w:.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

# Verificación con un batch real
imgs, lbls = next(iter(train_loader))
imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
logits = model(imgs)
loss = criterion(logits, lbls)
print("\nLogits shape:", logits.shape)
print("Labels shape:", lbls.shape)
print("Pérdida inicial:", loss.mean().item())

pos_weight por label:
  Atelectasis                    2.81
  Cardiomegaly                   2.97
  Consolidation                  15.25
  Edema                          4.73
  Enlarged Cardiomediastinum     21.96
  Fracture                       43.02
  Lung Lesion                    25.18
  Lung Opacity                   2.63
  No Finding                     1.51
  Pleural Effusion               2.36
  Pleural Other                  148.44
  Pneumonia                      8.62
  Pneumothorax                   15.61

Logits shape: torch.Size([32, 13])
Labels shape: torch.Size([32, 13])
Pérdida inicial: nan


In [14]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_total = 0.0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        mask = (lbls != -1).float()
        lbls_clean = torch.nan_to_num(lbls, nan=0.0).clamp(min=0)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = (criterion(logits, lbls_clean) * mask).mean()
        loss.backward()
        optimizer.step()

        loss_total += loss.item()

    return loss_total / len(loader)

In [15]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

def evaluate(model, loader, criterion, device, label_cols, umbral=0.5):
    model.eval()                       # modo evaluación (apaga dropout)
    loss_total = 0.0
    todas_probs, todas_lbls = [], []

    with torch.no_grad():              # no calcular gradientes
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss_total += criterion(logits, lbls).mean().item()  # ← aquí

            probs = torch.sigmoid(logits)
            todas_probs.append(probs.cpu())
            todas_lbls.append(lbls.cpu())

    # Juntar todos los batches
    y_prob = torch.cat(todas_probs).numpy()
    y_prob = np.nan_to_num(y_prob, nan=0.0)
    y_true = torch.cat(todas_lbls).numpy()
    
    # Reemplazar NaN en y_true con 0
    y_true = np.nan_to_num(y_true, nan=0.0)
    y_true = np.clip(y_true, 0, 1)
    y_pred = (y_prob >= umbral).astype(int)   # binarizar con el umbral

    # AUC por etiqueta (independiente del umbral)
    aucs = {}
    for i, nombre in enumerate(label_cols):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        aucs[nombre] = roc_auc_score(y_true[:, i], y_prob[:, i])

    # Precision, recall, F1 macro (dependen del umbral)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
        # Por etiqueta para el desglose
    prec_arr, rec_arr, f1_arr, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
        
        # TP, FP, FN, TN por etiqueta
    confusion_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        tn, fp, fn, tp = confusion_matrix(
            y_true[:, i], y_pred[:, i], labels=[0, 1]
        ).ravel()
        confusion_por_etiqueta[nombre] = {
            "TP": int(tp), "FP": int(fp),
            "FN": int(fn), "TN": int(tn)
        }
    # Métricas por etiqueta consolidadas
    metricas_por_etiqueta = {}
    for i, nombre in enumerate(label_cols):
        metricas_por_etiqueta[nombre] = {
            "auc":       aucs.get(nombre, float("nan")),
            "precision": float(prec_arr[i]),
            "recall":    float(rec_arr[i]),
            "f1":        float(f1_arr[i]),
            **confusion_por_etiqueta[nombre],   # TP, FP, FN, TN
        }

    metricas = {
        "loss": loss_total / len(loader),
        "auc": np.mean(list(aucs.values())) if aucs else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    return metricas, aucs, metricas_por_etiqueta   # <- ahora son DOS valores

In [16]:
import inspect
print("loss" in inspect.getsource(evaluate))

True


In [17]:
# Correr ANTES del loop MLflow
#model = ToraxCNN(n_classes=len(LABEL_COLS)).to(DEVICE)
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [18]:
NUM_EPOCHS = 20
mejor_auc = 0.0
mejor_epoch = 0
with mlflow.start_run(run_name="cnn-liviana3000"):

    # --- Registrar los parámetros del experimento (una sola vez) ---
    mlflow.log_params({
        "modelo": "ToraxCNN_liviana",
        "condicion": "crop",      # <- tu variable independiente
        "dataset": "df_subset_3000",
        "n_etiquetas": len(LABEL_COLS),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": NUM_EPOCHS,
        "img_size": IMG_SIZE,
        "umbral": 0.5,
    })


    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        metricas, aucs_detalle, metricas_por_etiqueta = evaluate(model, val_loader, criterion, DEVICE, LABEL_COLS)

        
        print(f"Época {epoch:2d}/{NUM_EPOCHS} | "
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {metricas['loss']:.4f} | "
            f"val_AUC: {metricas['auc']:.4f}")

        mlflow.log_metrics({
                "train_loss": train_loss,
                "val_loss": metricas["loss"],
                "val_auc": metricas["auc"],
                "val_precision": metricas["precision"],
                "val_recall": metricas["recall"],
                "val_f1": metricas["f1"],
            }, step=epoch)


        if metricas["auc"] > mejor_auc:
            mejor_auc = metricas["auc"]
            mejor_epoch = epoch
            torch.save(model.state_dict(), "mejor_modelo.pth")
            print(f"   ↑ nuevo mejor AUC: {mejor_auc:.4f} (época {mejor_epoch})")
        scheduler.step(metricas["auc"])
            
    model.load_state_dict(torch.load("mejor_modelo.pth"))
    _, aucs_final, metricas_por_etiqueta_final = evaluate(
        model, val_loader, criterion, DEVICE, LABEL_COLS
    )

        # AUC por etiqueta
    for label, auc_val in aucs_final.items():
        if not np.isnan(auc_val):
            mlflow.log_metric(f"auc_{label}", auc_val)

    # Precision, Recall, F1, TP, FP, FN, TN por etiqueta
    for label, vals in metricas_por_etiqueta_final.items():
        mlflow.log_metrics({
            f"precision_{label}": vals["precision"],
            f"recall_{label}":    vals["recall"],
            f"f1_{label}":        vals["f1"],
            f"TP_{label}":        vals["TP"],
            f"FP_{label}":        vals["FP"],
            f"FN_{label}":        vals["FN"],
            f"TN_{label}":        vals["TN"],
        })

    # Resumen del run
    mlflow.log_metrics({
        "mejor_val_auc": mejor_auc,
        "mejor_epoch":   mejor_epoch,
    })

    # CSV artifact consolidado
    df_por_etiqueta = pd.DataFrame(metricas_por_etiqueta_final).T
    df_por_etiqueta.to_csv("metricas_por_etiqueta.csv")
    mlflow.log_artifact("metricas_por_etiqueta.csv")
    mlflow.log_artifact("mejor_modelo.pth")

print(f"\nListo. Mejor AUC: {mejor_auc:.4f} en época {mejor_epoch}")


2026/07/14 11:50:30 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Época  1/20 | train_loss: 1.0664 | val_loss: nan | val_AUC: 0.4785
   ↑ nuevo mejor AUC: 0.4785 (época 1)
Época  2/20 | train_loss: 1.0304 | val_loss: nan | val_AUC: 0.5260
   ↑ nuevo mejor AUC: 0.5260 (época 2)
Época  3/20 | train_loss: 1.0344 | val_loss: nan | val_AUC: 0.5270
   ↑ nuevo mejor AUC: 0.5270 (época 3)
Época  4/20 | train_loss: 1.0181 | val_loss: nan | val_AUC: 0.5652
   ↑ nuevo mejor AUC: 0.5652 (época 4)
Época  5/20 | train_loss: 1.0124 | val_loss: nan | val_AUC: 0.5760
   ↑ nuevo mejor AUC: 0.5760 (época 5)
Época  6/20 | train_loss: 1.0096 | val_loss: nan | val_AUC: 0.5719
Época  7/20 | train_loss: 1.0112 | val_loss: nan | val_AUC: 0.5954
   ↑ nuevo mejor AUC: 0.5954 (época 7)
Época  8/20 | train_loss: 1.0031 | val_loss: nan | val_AUC: 0.5928
Época  9/20 | train_loss: 0.9951 | val_loss: nan | val_AUC: 0.6032
   ↑ nuevo mejor AUC: 0.6032 (época 9)
Época 10/20 | train_loss: 1.0005 | val_loss: nan | val_AUC: 0.6007
Época 11/20 | train_loss: 0.9960 | val_loss: nan | val_AU

In [19]:
# --- Print final del run ---
print(f"\n{'='*60}")
print(f"  RESULTADOS FINALES — {mejor_epoch} épocas")
print(f"{'='*60}")
print(f"  Mejor época:  {mejor_epoch}")
print(f"  Mejor AUC:    {mejor_auc:.4f}")
print(f"{'='*60}")
print(f"\n{'Etiqueta':<30} {'AUC':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'TN':>4}")
print(f"{'-'*74}")
for label, vals in metricas_por_etiqueta_final.items():
    auc_str = f"{vals['auc']:.4f}" if not np.isnan(vals['auc']) else "  nan"
    print(f"  {label:<28} {auc_str:>6} {vals['precision']:>6.4f} {vals['recall']:>6.4f} {vals['f1']:>6.4f} {vals['TP']:>4} {vals['FP']:>4} {vals['FN']:>4} {vals['TN']:>4}")
print(f"{'='*60}\n")


  RESULTADOS FINALES — 20 épocas
  Mejor época:  20
  Mejor AUC:    0.6235

Etiqueta                          AUC   Prec Recall     F1   TP   FP   FN   TN
--------------------------------------------------------------------------
  Atelectasis                  0.6579 0.3628 0.3388 0.3504   41   72   80  400
  Cardiomegaly                 0.6804 0.3212 0.4000 0.3563   44   93   66  390
  Consolidation                0.5870 0.0971 0.3226 0.1493   10   93   21  469
  Edema                        0.7336 0.2313 0.4928 0.3148   34  113   35  411
  Enlarged Cardiomediastinum   0.5731 0.0667 0.0417 0.0513    1   14   23  555
  Fracture                     0.5960 0.0000 0.0000 0.0000    0   23    6  564
  Lung Lesion                  0.5878 0.0317 0.2857 0.0571    4  122   10  457
  Lung Opacity                 0.6053 0.3026 0.1643 0.2130   23   53  117  400
  No Finding                   0.7355 0.5548 0.4456 0.4943   86   69  107  331
  Pleural Effusion             0.7075 0.4435 0.3873 0.4135

In [24]:
model.load_state_dict(torch.load("mejor_modelo.pth"))
metricas_05, _, _ = evaluate(model, val_loader, criterion, DEVICE, LABEL_COLS, umbral=0.9)
metricas_03, _, _ = evaluate(model, val_loader, criterion, DEVICE, LABEL_COLS, umbral=0.5)
metricas_02, _, _ = evaluate(model, val_loader, criterion, DEVICE, LABEL_COLS, umbral=0.1)

print("="*70)
print("CNN SIMPLE + CROP — Análisis de Umbrales")
print("="*70)
print(f"umbral=0.9 → AUC: {metricas_05['auc']:.4f} | Precision: {metricas_05['precision']:.4f} | Recall: {metricas_05['recall']:.4f} | F1: {metricas_05['f1']:.4f}")
print(f"umbral=0.5 → AUC: {metricas_03['auc']:.4f} | Precision: {metricas_03['precision']:.4f} | Recall: {metricas_03['recall']:.4f} | F1: {metricas_03['f1']:.4f}")
print(f"umbral=0.1 → AUC: {metricas_02['auc']:.4f} | Precision: {metricas_02['precision']:.4f} | Recall: {metricas_02['recall']:.4f} | F1: {metricas_02['f1']:.4f}")
print("="*70)

CNN SIMPLE + CROP — Análisis de Umbrales
umbral=0.9 → AUC: 0.6235 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
umbral=0.5 → AUC: 0.6235 | Precision: 0.1893 | Recall: 0.2328 | F1: 0.1903
umbral=0.1 → AUC: 0.6235 | Precision: 0.1203 | Recall: 1.0000 | F1: 0.2006


In [21]:
import pandas as pd
print(pd.Series(aucs_detalle).sort_values(ascending=False))

No Finding                    0.735531
Edema                         0.733599
Pleural Effusion              0.707536
Cardiomegaly                  0.680444
Pneumothorax                  0.660777
Atelectasis                   0.657918
Lung Opacity                  0.605298
Fracture                      0.595968
Lung Lesion                   0.587836
Consolidation                 0.587016
Enlarged Cardiomediastinum    0.573081
Pneumonia                     0.562345
Pleural Other                 0.418027
dtype: float64
